# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a complete, step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using a [Croissant schema](https://mlcommons.org/croissant/) accessible at the following URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")


## 2. Data Overview
Review available record sets, their fields, and the `@id`s for all components.

Entities are referenced **only by their `@id` properties** for clarity and reproducibility.

In [ ]:
# Examine all available record sets and their fields
print('Available record sets and their fields:')
record_sets = []
for record_set in dataset.record_sets:
    print(f"  - Record set: {record_set['@id']}  ({record_set.get('name')})")
    record_sets.append(record_set['@id'])
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]  # normalize to list
    print("    Fields:")
    for field in fields:
        if isinstance(field, dict):
            print(f"      - {field['@id']}  ({field.get('name', '')})")
        else:
            print(f"      - {field}")
    print()
if not record_sets:
    print('-- No direct record sets declared at root. Scanning dataset for available tables... --')
    # Try detection by file objects
    files = getattr(metadata, 'distribution', [])
    # Not all Croissant files expose .record_sets, so try via files
    for f in files:
        print(f"  Found DataFileObject: {f['@id']}")

## 3. Data Extraction
Load data from available record sets. All record sets and fields are referenced by their `@id` properties as previously shown.

> **Note:** If no record sets were found above, try extracting from likely tables or top-level objects.

In [ ]:
# We'll try to extract from the main data table (most likely the only/top record set)
dataframes = {}

# If record sets discovered, extract them; otherwise, attempt all possible tables
if record_sets:
    extract_record_sets = record_sets
else:
    # Try a default record set id based on typical Croissant usage
    extract_record_sets = ['cr:RecordSet']
    print(f"Assuming main record set: {extract_record_sets}")

for record_set_id in extract_record_sets:
    try:
        # Yields dicts, keys are field @id
        records = list(dataset.records(record_set=record_set_id))
        if not records:
            print(f'No records found for record set {record_set_id}')
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Could not extract from {record_set_id}: {e}")


## 4. Exploratory Data Analysis (EDA)
Apply processing to a selected numeric field (referenced via its `@id`), such as filtering, normalization, and grouping.

In [ ]:
# --- Choose your record set and numeric field @id from previous step ---
chosen_record_set = next(iter(dataframes.keys())) # First loaded
df = dataframes[chosen_record_set]

# Pick a numeric field: here, we try to choose one with a numeric-sounding name, or you can set it manually
possible_numeric_fields = [c for c in df.columns if any(s in c.lower() for s in ['abundance', 'intensity', 'count', 'quantity', 'coverage', 'mw', 'weight', 'score'])]
numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df.select_dtypes('number').columns[0]

print(f'Using numeric field: {numeric_field}')

# Filter for rows where value > threshold
threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records")

# Normalization
filtered_df[f'{numeric_field}_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

# Group by a categorical field if possible (e.g. 'description' or similar)
possible_group_fields = [c for c in df.columns if c != numeric_field and df[c].nunique() < len(df)//2 and df[c].dtype==object]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (showing means):")
    print(grouped_df.head())
else:
    print('No suitable group field found.')

## 5. Visualization
Visualize the distribution of the selected numeric field, and (if available) its grouping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

if group_field:
    plt.figure(figsize=(10, 4))
    # Bar plot of group means
    grp = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
    sns.barplot(x=grp.index, y=grp.values)
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.ylabel(f'Mean {numeric_field}')
    plt.xlabel(group_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded the FAIR^2 Croissant dataset and explored its metadata
- Inspected available record sets and fields by their `@id`
- Loaded records into DataFrames and performed filtering, normalization, and grouping on numeric fields
- Visualized distributions and categorical means

You can now proceed with further analysis or machine learning based on the extracted data using the `mlcroissant` library.
